# 第十二课:Backpropagation——梯度怎么往回爬

讲义:[12-backprop.md](12-backprop.md)。

## 1. 必考结论一:softmax+CE 梯度 = p − onehot(讲义 §2)

In [ ]:
import numpy as np

def softmax(x):
    m = np.max(x); e = np.exp(x - m)
    return e / e.sum()

z, t = np.array([2.0, 1.0, 0.1]), 0
analytic = softmax(z) - np.eye(3)[t]        # p - onehot

eps = 1e-6                                  # 数值梯度当证人
numeric = []
for i in range(3):
    zp, zm = z.copy(), z.copy()
    zp[i] += eps; zm[i] -= eps
    lp = -np.log(softmax(zp)[t]); lm = -np.log(softmax(zm)[t])
    numeric.append((lp - lm) / (2 * eps))

print('解析梯度:', np.round(analytic, 6))
print('数值梯度:', np.round(numeric, 6))
print('最大误差:', float(np.abs(analytic - np.array(numeric)).max()))

## 2. 必考结论二:两层 MLP 手写反向 + 梯度检查(讲义 §3–5)

In [ ]:
rng = np.random.default_rng(7)
W1, W2 = rng.normal(size=(3, 5)), rng.normal(size=(5, 3)) * 0.5
x, y = rng.normal(size=5), rng.normal(size=5)
eps = 1e-6

def mlp_loss(W1, W2):
    h = np.maximum(W1 @ x, 0)
    return 0.5 * ((W2 @ h - y) ** 2).mean()

h = np.maximum(W1 @ x, 0); o = W2 @ h
dW2 = np.outer(o - y, h) / len(y)          # 模板1: 外积(梯度, 输入)
dh  = W2.T @ (o - y) / len(y)              # 模板2: Wᵀ × 梯度
dW1 = np.outer(dh * (h > 0), x)            # ReLU 门卫

num = np.zeros_like(W1)                    # 有限差分证人
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        Wp, Wm = W1.copy(), W1.copy()
        Wp[i, j] += eps; Wm[i, j] -= eps
        num[i, j] = (mlp_loss(Wp, W2) - mlp_loss(Wm, W2)) / (2 * eps)

print('loss =', round(mlp_loss(W1, W2), 4))
print('dW1 解析 vs 数值 最大误差 =', float(np.abs(dW1 - num).max()))

## 作业现场

- C3:`softmax_ce_grad(logits, target)` + 数值验证。
- E1:label smoothing 版梯度。
- D1:梯度范数监控器。

In [ ]:
# TODO(C3): 直接版梯度
def softmax_ce_grad(logits, target):
    # 返回 p - onehot(不许用 autograd)
    ...

# 数值验证同 cell 1 方法

# TODO(D1): 每层梯度范数监控(伪生产)
# def grad_norm_report(grads: dict):  打印每层名字和 L2 范数,标记 >10 的层
